In [ ]:
# 2026-10-20
from django.core.exceptions import ObjectDoesNotExist
from csae_pyutils import gsheet_to_df
from tqdm import tqdm
from apis_core.apis_metainfo.models import Collection, Uri
from apis_core.apis_entities.models import Person, Event
from apis_core.apis_vocabularies.models import PersonEventRelation
from apis_core.apis_relations.models import PersonEvent

In [ ]:
col = Collection.objects.filter(name__startswith="Brahms Online").first()
domain = "brahms-online"
person_event_rel = PersonEventRelation.objects.get(id="1256")
person_event_rel, col, domain



In [ ]:
df = gsheet_to_df("1KdT0HXkrIg_U5t-PkSnspUa4amIW4nKflR63hg3ytcc")

In [ ]:
df.head(3)

In [ ]:
events = {}
persons = {}
for i, row in tqdm(df.iterrows(), total=len(df)):
    tmp_id = row["event_id"].strip()
    pmb_uri = f"https://brahms-online.oeaw.ac.at/{tmp_id}.html"
    pmb_uri = Uri.objects.get(uri=pmb_uri)
    try:
        event = events[tmp_id]
    except KeyError:
        event = Event.objects.get(id=pmb_uri.entity.id)
        events[tmp_id] = event
    person_id = row["person_id"]
    person_url = f"https://brahms-online.oeaw.ac.at/personenregister-id-{person_id:0>4}.html"
    try:
        person = persons[person_id]
    except KeyError:
        try:
            person_id = Uri.objects.get(uri=person_url).entity.id
        except ObjectDoesNotExist:
            continue
        person = Person.objects.get(id=person_id)
        persons[person_id] = person
    PersonEvent.objects.get_or_create(
        related_person=person, related_event=event, relation_type=person_event_rel, start_date_written=event.start_date_written, end_date_written=event.start_date_written
    )
    # item = {
    #     "name": row["nachname"][:254] if pd.notna(row["nachname"]) else "",
    #     "first_name": row["vorname"][:254] if pd.notna(row["vorname"]) else "",
    # }
     